# LayaMed on Colab: from the base model to the interface

A complete, generic example with **synthetic cases** (generated by rules, no patient data at all):

1. environment and GPU;
2. synthetic cases of the 4 diseases (appendicitis, cholecystitis, pancreatitis, diverticulitis), complete and partial;
3. fine-tuning Laya Multilingual (mmBERT encoder) on two questions: diagnosis (`choice`) and management (`noul`);
4. evaluation on the test set, by case stage;
5. out-of-scope detector (non-clinical text gets no probability);
6. recalibration by the amount of text written (optional);
7. HTML report;
8. server + LayaMed interface opened inside Colab;
9. download of the checkpoint to run it on your own computer.

**Before you start:** `Runtime → Change runtime type → GPU` (a T4 is enough; A100 or L4 are faster).

**The numbers in this notebook have no clinical value.** Rule-based cases are much easier than real records, and the management label is a teaching rule per disease. The goal is to show the full path; for a real model, replace the synthetic cases with your own data (see `docs/02-data.md`) and respect that data's usage agreement.

Approximate time on a T4: 25–35 min (mostly training). To just test the path, write `QUICK = True` in cell 1 (or set the environment variable `LAYAMED_QUICK=1` before starting Jupyter locally).

In [ ]:
# 1. Environment: GPU, packages and repository
import os, subprocess, sys
from pathlib import Path

QUICK = os.environ.get("LAYAMED_QUICK") == "1"   # write QUICK = True here for few cases and 1 epoch (only to test the path)
REPO_URL = "https://github.com/csbl-usp/LayaMed.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    ROOT = Path("/content/LayaMed")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", "-q", REPO_URL, str(ROOT)], check=True)
    # Colab already has torch, transformers, safetensors, huggingface_hub, numpy, pandas and plotly.
    # --no-deps on laya: pip does not replace Colab's CUDA build of torch.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "laya==0.4.7", "--no-deps"], check=True)
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import torch  # torch before numpy/pandas (see docs/01-environment.md, OpenMP note)
import laya

assert laya.__version__ == "0.4.7", "use laya==0.4.7: other versions build the sequence differently"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
assert DEVICE == "cuda", "enable the GPU: Runtime → Change runtime type → GPU"
GPU = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
BF16 = torch.cuda.get_device_capability(0)[0] >= 8   # Ampere or newer; the T4 uses fp16
print(f"Repository: {ROOT} | laya {laya.__version__} | {GPU} ({VRAM_GB:.0f} GB) | {'bf16' if BF16 else 'fp16'}"
      f" | {'QUICK' if QUICK else 'full'} mode")

## 2. Synthetic cases

`synthetic/generate_cases.py` builds, from textbook rules, 250 base cases per disease. Each base case becomes 4 rows, in the order a case is written: `hpi_partial` (start of the history), `hpi` (history), `hpi_pe` (history + exam) and `complete` (+ labs). The train/val/test split (70/15/15) is per base case, so the versions of one case never land in different sets.

The `model_text` column is already in the format Laya reads (`layamed.inference.build_text`):

```
HISTORY OF PRESENT ILLNESS:
...

PHYSICAL EXAMINATION:
...

PERTINENT / ABNORMAL LABS:
...
```

**Why train on partial cases too:** a model trained only on complete cases answers with high confidence even when almost nothing has been written. With the partial versions and the true label, it learns to give moderate probabilities while the case is incomplete.

In [ ]:
# 2. Generate the CSVs and join the 4 diseases
import random
import numpy as np
import pandas as pd
from layamed import inference as inf

subprocess.run([sys.executable, "synthetic/generate_cases.py"], check=True)
cases = pd.concat([pd.read_csv(f"data/synthetic/partial_cases_{c}.csv", keep_default_na=False) for c in inf.CLASSES],
                  ignore_index=True)
if QUICK:  # 12 base cases per disease and split
    ids = cases.groupby(["disease", "split"])["case_id"].apply(lambda s: list(dict.fromkeys(s))[:12]).explode()
    cases = cases[cases["case_id"].isin(set(ids))].reset_index(drop=True)

# TEACHING management label (one rule per disease, only to show the 2nd question; not a clinical criterion):
# appendicitis and cholecystitis → surgical; pancreatitis and diverticulitis → conservative.
SURGICAL = {"appendicitis": 1, "cholecystitis": 1, "pancreatitis": 0, "diverticulitis": 0}
cases["surgical"] = cases["disease"].map(SURGICAL)
print(cases.groupby(["split", "stage"]).size().unstack())
print("\nExample (complete case):\n")
print(cases.query("stage == 'complete'").iloc[0]["model_text"])

## 3. Fine-tuning

Laya is an encoder (mmBERT, 322 M parameters) with a head that reads **the question and the options together with the text**. Each training item is a `question + options | record` sequence built by `laya.common.build_sequence` (the same function inference uses).

- Questions: the same as `layamed.inference.QUESTIONS` (diagnosis with 4 options; yes/no management).
- Loss: `laya.common.proper_reward`, a strictly proper scoring rule (it rewards calibrated probabilities), with a smoothed target (0.95 on the right answer).
- AdamW: encoder 3e-5, head 1e-4; 3 epochs; the saved epoch is the one with the lowest diagnosis NLL on validation.
- After training, one temperature per question type, chosen by grid search on validation.

The base model comes from Hugging Face (`convaiinnovations/laya`, `multilingual` folder, ~1.3 GB).

In [ ]:
# 3a. Base model and training items
import json, time
from huggingface_hub import snapshot_download
from laya.common import build_sequence, build_model, proper_reward, collate_items, QTYPES, TEMP_MIN, TEMP_MAX
from safetensors.torch import load_file, save_file
from transformers import AutoTokenizer

SEED = 42
P_TARGET = 0.95
EPOCHS = 1 if QUICK else 3
LR_ENCODER, LR_HEAD = 3e-5, 1e-4
BIG_GPU = VRAM_GB >= 20
BATCH_SIZE, GRAD_ACCUM = (8, 1) if BIG_GPU else (4, 2)   # effective batch of 8 in both cases
EVAL_BATCH = 32 if BIG_GPU else 16
CHECKPOINT = ROOT / "checkpoint"


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


BASE_DIR = Path(snapshot_download("convaiinnovations/laya", allow_patterns=["multilingual/*"], token=False)) / "multilingual"
BASE_CFG = json.loads((BASE_DIR / "rl_agent_config.json").read_text(encoding="utf-8"))
tok = AutoTokenizer.from_pretrained(str(BASE_DIR / "tokenizer"))
print("Base model:", BASE_DIR)


def item(text, question, correct):
    k = len(inf.QUESTIONS[question]["criteria"])
    ids, markers = build_sequence(tok, text, inf.Q_INTERNAL[question], inf.MAX_LEN, inf.HEAD_MAX_LEN)
    target = [P_TARGET if i == correct else (1 - P_TARGET) / (k - 1) for i in range(k)]
    return {"ids": ids, "markers": markers, "qtype": QTYPES[inf.QUESTIONS[question]["type"]],
            "target": target, "label": correct}


def items(df):
    # Diagnosis: index of the disease in CLASSES. Management (noul): options in the order "false", "true" → 1 = surgical.
    out = {"diag": [], "management": []}
    for text, y, surg in df[["model_text", "label_id", "surgical"]].itertuples(index=False):
        out["diag"].append(item(text, "diagnosis", int(y)))
        out["management"].append(item(text, "management", int(surg)))
    return out


ITEMS = {s: items(cases[cases["split"] == s]) for s in ("train", "val")}
train = ITEMS["train"]["diag"] + ITEMS["train"]["management"]
print(f"Training items: {len(train)} | validation: {len(ITEMS['val']['diag'])} per question | "
      f"tokens per item: median {int(np.median([len(i['ids']) for i in train]))}")

In [ ]:
# 3b. Training loop
AMP_DTYPE = torch.bfloat16 if BF16 else torch.float16


def make_batch(lst):
    b = collate_items([lst], tok.pad_token_id)
    return {k: v.to(DEVICE) for k, v in b.items() if torch.is_tensor(v)}


def forward(model, b):
    return model(b["input_ids"], b["attention_mask"], b["marker_pos"], b["marker_mask"], b["qtype"])


@torch.no_grad()
def logits(model, lst):
    """Logits [n, k] of items with the same number of options k, in list order."""
    model.eval()
    k = len(lst[0]["target"])
    order = sorted(range(len(lst)), key=lambda i: len(lst[i]["ids"]))
    parts = []
    for i in range(0, len(order), EVAL_BATCH):
        b = make_batch([lst[j] for j in order[i:i + EVAL_BATCH]])
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            lg, _ = forward(model, b)
        parts.append(lg[:, :k].float().cpu())
    lg = torch.cat(parts)
    out = torch.empty_like(lg)
    out[torch.tensor(order)] = lg
    return out


def nll(lg, y, T=1.0):
    return float(torch.nn.functional.cross_entropy(lg / T, torch.tensor(y)))


set_seed(SEED)
model = build_model(BASE_CFG, encoder_dir=str(BASE_DIR / "encoder"), pretrained=False)
model.load_state_dict(load_file(str(BASE_DIR / "model.safetensors")), strict=True)
model.to(DEVICE)
if not BIG_GPU and hasattr(model.encoder, "gradient_checkpointing_enable"):
    model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
enc = [p for n, p in model.named_parameters() if n.startswith("encoder.")]
head = [p for n, p in model.named_parameters() if not n.startswith("encoder.")]
opt = torch.optim.AdamW([{"params": enc, "lr": LR_ENCODER}, {"params": head, "lr": LR_HEAD}], weight_decay=0.01)
scaler = torch.amp.GradScaler("cuda", enabled=not BF16)   # fp16 needs loss scaling; bf16 does not

y_val = [i["label"] for i in ITEMS["val"]["diag"]]
best, history, t0 = (float("inf"), 0, None), [], time.time()
for ep in range(1, EPOCHS + 1):
    random.shuffle(train)
    model.train()
    opt.zero_grad()
    n_steps, total = (len(train) + BATCH_SIZE - 1) // BATCH_SIZE, 0.0
    for step, start in enumerate(range(0, len(train), BATCH_SIZE)):
        b = make_batch(train[start:start + BATCH_SIZE])
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            lg, _ = forward(model, b)
        loss = -proper_reward(torch.softmax(lg.float(), -1), b["target"], b["qtype"], b["marker_mask"].float()).mean()
        scaler.scale(loss / GRAD_ACCUM).backward()
        if (step + 1) % GRAD_ACCUM == 0 or step + 1 == n_steps:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); opt.zero_grad()
        total += loss.item() * len(b["input_ids"])
        if (step + 1) % 200 == 0:
            print(f"  epoch {ep} · step {step + 1}/{n_steps} · {(time.time() - t0) / 60:.1f} min", flush=True)
    lg_val = logits(model, ITEMS["val"]["diag"])
    acc = float((lg_val.argmax(1) == torch.tensor(y_val)).float().mean() * 100)
    history.append({"epoch": ep, "train_loss": total / len(train), "val_nll": nll(lg_val, y_val), "val_accuracy_%": acc})
    print(history[-1], flush=True)
    if history[-1]["val_nll"] < best[0]:
        best = (history[-1]["val_nll"], ep, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()})
model.load_state_dict(best[2])
print(f"Chosen epoch: {best[1]} | training took {(time.time() - t0) / 60:.1f} min")

In [ ]:
# 3c. Temperatures (grid search on validation) and a checkpoint in the format laya.Agent loads
import copy, shutil


def temperature(lg, y):
    grid = torch.exp(torch.linspace(np.log(TEMP_MIN), np.log(TEMP_MAX), 400))
    return float(grid[torch.stack([torch.nn.functional.cross_entropy(lg / t, torch.tensor(y)) for t in grid]).argmin()])


TEMPS = {"choice": temperature(logits(model, ITEMS["val"]["diag"]), y_val),
         "noul": temperature(logits(model, ITEMS["val"]["management"]), [i["label"] for i in ITEMS["val"]["management"]])}
cfg = copy.deepcopy(BASE_CFG)
cfg["max_len"], cfg["head_max_len"] = inf.MAX_LEN, inf.HEAD_MAX_LEN
temps = list(cfg.get("temperature", [1.0, 1.0, 1.0]))
for qtype, t in TEMPS.items():
    temps[QTYPES[qtype]] = t
cfg.update({"temperature": temps, "temperature_by_options": {}, "lang_temperatures": {}})

if CHECKPOINT.exists():
    shutil.rmtree(CHECKPOINT)
shutil.copytree(BASE_DIR, CHECKPOINT, ignore=shutil.ignore_patterns("*.safetensors", "*.bin", "*.pt"))
save_file({k: v.contiguous() for k, v in model.state_dict().items()}, str(CHECKPOINT / "model.safetensors"))
model.encoder.config.save_pretrained(str(CHECKPOINT / "encoder"))
tok.save_pretrained(str(CHECKPOINT / "tokenizer"))
(CHECKPOINT / "rl_agent_config.json").write_text(json.dumps(cfg, indent=2), encoding="utf-8")
(CHECKPOINT / "training_meta.json").write_text(json.dumps(
    {"data": "synthetic cases (synthetic/generate_cases.py)", "laya": laya.__version__, "seed": SEED,
     "epoch": best[1], "temperatures": TEMPS, "quick": QUICK, "history": history}, indent=2), encoding="utf-8")
print("Temperatures:", {k: round(v, 3) for k, v in TEMPS.items()}, "| checkpoint:", CHECKPOINT)
del model, opt
torch.cuda.empty_cache()

## 4. Test evaluation

From here on everything goes through the real interface path: `layamed.inference.LayaInference` loads the saved checkpoint with `laya.Agent`, exactly as the server will.

In [ ]:
# 4. Accuracy by case stage (test) and mean probability of the right disease
import plotly.graph_objects as go
from plotly.subplots import make_subplots

laya_model = inf.LayaInference(CHECKPOINT, batch_size=16)
test = cases[cases["split"] == "test"].reset_index(drop=True)
P, p_surg = laya_model.predict(test["model_text"].tolist())
test["correct"] = P.argmax(1) == test["label_id"].to_numpy()
test["p_right"] = P[np.arange(len(test)), test["label_id"].to_numpy()]
test["management_ok"] = (p_surg >= 0.5) == test["surgical"].astype(bool).to_numpy()
STAGES = ["hpi_partial", "hpi", "hpi_pe", "complete"]
by_stage = test.groupby("stage")[["correct", "p_right", "management_ok"]].mean().reindex(STAGES) * 100
print(by_stage.round(1))

fig_eval = make_subplots(rows=1, cols=2, subplot_titles=("Diagnosis accuracy by stage (%)",
                                                         "Mean probability of the right disease (%)"))
fig_eval.add_bar(x=STAGES, y=by_stage["correct"], marker_color="#2E9461", row=1, col=1, name="accuracy")
fig_eval.add_bar(x=STAGES, y=by_stage["p_right"], marker_color="#1F6E9C", row=1, col=2, name="P(right)")
fig_eval.update_yaxes(range=[0, 100])
fig_eval.update_layout(template="plotly_white", showlegend=False, height=380,
                       title="Test (synthetic cases): the model grows more confident as the case is written")
fig_eval.show()

## 5. Out-of-scope detector

Laya is closed on 4 classes: without a filter, a cake recipe also comes out as "appendicitis 80%". The detector (`layamed/scope.py`) requires two signals:

1. **distance** (Mahalanobis) from the text vector in the Laya encoder to the training cases; threshold = 1.10 × the largest distance among valid validation texts;
2. **clinical vocabulary**: at least N clinical terms (symptoms, exam, labs, units); N = 0.1% quantile of validation.

The thresholds come only from valid texts. The non-medical texts in `synthetic/out_of_scope.py` (hand-written) are only used to check.

In [ ]:
# 5. Fit, calibration and check
from layamed.scope import ScopeDetector, count_clinical_terms
from layamed.service import EXAMPLES, MIN_CONTEXT_TOKENS
from layamed import recalibration as rc
from synthetic.out_of_scope import NON_MEDICAL_LONG, OTHER_DISEASES


def with_context(texts):
    # Only texts that would pass the interface's minimum context (below it Laya does not even run)
    return [t for t in texts if laya_model.budget(t)["diagnosis"]["record_tokens_used"] >= MIN_CONTEXT_TOKENS]


train_complete = cases.query("split == 'train' and stage == 'complete'")["model_text"].tolist()
scope = ScopeDetector.fit(laya_model.embeddings(train_complete))
valid_texts = with_context(cases.query("split == 'val'")["model_text"].tolist()
                           + [p for ex in EXAMPLES for p in rc.prefixes(inf.build_text(*ex), (0.5, 0.8, 1.0))])
scope.calibrate(scope.distance(laya_model.embeddings(valid_texts)), [count_clinical_terms(t) for t in valid_texts])

groups = {"valid (test)": with_context(test["model_text"].tolist()),
          "non-medical": with_context([inf.build_text(t, "", "") for t in NON_MEDICAL_LONG]),
          "other diseases (MI, pneumonia...)": with_context([inf.build_text(*m) for m in OTHER_DISEASES])}
for name, texts in groups.items():
    ev = scope.evaluate(laya_model.embeddings(texts), texts)
    print(f"{name:36s} {len(texts):4d} texts | blocked: {100 * np.mean([e['out_of_scope'] for e in ev]):5.1f}%")
print(f"\nThresholds: distance <= {scope.limit:.1f}; clinical terms >= {scope.min_terms}")
print("Expected: ~0% of valid texts blocked and ~100% of non-medical ones. Other diseases are NOT blocked: the detector"
      " separates 'clinical case' from 'any text'; separating 'another disease' needs a trained 5th option.")
(ROOT / "data").mkdir(exist_ok=True)
scope.save(ROOT / "data" / "scope.pt")

## 6. Recalibration by the amount of text written (optional)

Since the model was also trained on partial cases, it already moderates its probabilities while the case is incomplete. Recalibration (`layamed/recalibration.py`) is an extra layer, mostly useful for models trained only on complete cases: it mixes the Laya output with the class frequencies, with a weight that grows with the text written

`w = sigmoid(a + b·log(1 + typed tokens) + e·[exam] + g·[labs])`, `logit = w·log P + (1 − w)·log freq`.

The 4 parameters are fitted on prefixes (cuts from 0% to 100% of the words) of the complete validation cases. The decision to use it is also taken on validation, never on test: the cell fits on half of the validation cases and keeps the recalibration only if it improves the NLL on the other half; then it refits on the whole validation set and saves the file. Otherwise the interface uses the Laya output as is. The test NLL is only reported.

In [ ]:
# 6. Fit and decide on validation (two halves by base case); the test set is only reported
def prefixes_of(df):
    texts, y = [], []
    for t, label in df.query("stage == 'complete'")[["model_text", "label_id"]].itertuples(index=False):
        for p in rc.prefixes(t):
            texts.append(p); y.append(int(label))
    return texts, np.asarray(y)


def tokens(texts):
    return np.asarray([laya_model.budget(t)["diagnosis"]["record_tokens_used"] for t in texts])


nll_np = lambda P, y: float(-np.log(np.clip(P[np.arange(len(y)), y], 1e-9, 1)).mean())
freq = cases.query("split == 'train' and stage == 'complete'")["label_id"].value_counts(normalize=True).sort_index().to_numpy()
n_empty = int(tokens([inf.build_text("", "", "")])[0])

val = cases[cases["split"] == "val"]
val_ids = sorted(val["case_id"].unique())
random.Random(SEED).shuffle(val_ids)
half = set(val_ids[: len(val_ids) // 2])
tf, yf = prefixes_of(val[val["case_id"].isin(half)])
tc, yc = prefixes_of(val[~val["case_id"].isin(half)])
Pf, _ = laya_model.predict(tf)
Pc, _ = laya_model.predict(tc)
recal_half = rc.Recalibrator.fit(Pf, tokens(tf), tf, yf, freq, n_empty)
nll_laya, nll_recal = nll_np(Pc, yc), nll_np(recal_half.predict(Pc, tokens(tc), tc), yc)
print(f"Validation NLL (prefixes, held-out half): Laya {nll_laya:.3f} | recalibrated {nll_recal:.3f}")

recal_file = ROOT / "data" / "recalibration.json"
if nll_recal < nll_laya:
    tv, yv, Pv = tf + tc, np.concatenate([yf, yc]), np.vstack([Pf, Pc])
    recal = rc.Recalibrator.fit(Pv, tokens(tv), tv, yv, freq, n_empty)   # final fit on the whole validation set
    recal.save(recal_file); print("Recalibration saved:", recal_file)
    tt, yt = prefixes_of(test)
    Pt, _ = laya_model.predict(tt)
    print(f"Test NLL (prefixes, report only): Laya {nll_np(Pt, yt):.3f} | recalibrated "
          f"{nll_np(recal.predict(Pt, tokens(tt), tt), yt):.3f}")
else:
    recal_file.unlink(missing_ok=True); print("Recalibration does not help this model: the interface uses Laya as is.")

## 7. HTML report

Evaluation and detector charts in a single file (`reports/layamed_colab.html`), to keep with the checkpoint.

In [ ]:
# 7. Report
fig_scope = go.Figure()
for name, texts in groups.items():
    fig_scope.add_box(y=scope.distance(laya_model.embeddings(texts)).numpy(), name=name, boxpoints="all", jitter=0.4)
fig_scope.add_hline(y=scope.limit, line_dash="dash", line_color="#C9A227", annotation_text="threshold")
fig_scope.update_layout(template="plotly_white", height=400, showlegend=False,
                        title="Distance to the training cases (above the threshold = out of scope)")
fig_scope.show()
(ROOT / "reports").mkdir(exist_ok=True)
with open(ROOT / "reports" / "layamed_colab.html", "w", encoding="utf-8") as f:
    f.write("<html><head><meta charset='utf-8'><title>LayaMed · example report</title></head><body style='font-family:sans-serif'>")
    f.write(f"<h2>LayaMed · example with synthetic cases</h2><p>{GPU} · laya {laya.__version__} · epochs {EPOCHS} · "
            f"chosen epoch {best[1]} · temperatures {json.dumps({k: round(v, 3) for k, v in TEMPS.items()})}</p>")
    f.write(by_stage.round(1).to_html())
    f.write(fig_eval.to_html(full_html=False, include_plotlyjs="cdn"))
    f.write(fig_scope.to_html(full_html=False, include_plotlyjs=False))
    f.write("<p>Rule-based cases: numbers without clinical value.</p></body></html>")
print("Report:", ROOT / "reports" / "layamed_colab.html")
del laya_model
torch.cuda.empty_cache()

## 8. LayaMed interface

`server.py` (FastAPI) loads the checkpoint, the detector and the recalibration, serves `web/index.html` and answers `POST /api/analyze`. On Colab, the server runs in the background and the page opens through a Colab address that only you can see (the cell sets `LAYAMED_ALLOWED_HOSTS=*`, because the Colab proxy sends its own host name).

On the page: write the case in a single field, with the titles `History of present illness:`, `Physical examination:` and `Labs:` at the start of lines (without titles, everything is read as history), or click an example case to watch it being typed word by word.

In [ ]:
# 8. Start the server and open the interface
import urllib.request

PORT = int(os.environ.get("LAYAMED_PORT", 8000))
server = subprocess.Popen([sys.executable, "server.py"], cwd=ROOT,
                          env={**os.environ, "PORT": str(PORT), "PYTHONIOENCODING": "utf-8",
                               **({"LAYAMED_ALLOWED_HOSTS": "*"} if IN_COLAB else {})},
                          stdout=open(ROOT / "server.log", "w", encoding="utf-8"), stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/config", timeout=1); break
    except Exception:
        if server.poll() is not None:
            raise RuntimeError("the server stopped:\n" + (ROOT / "server.log").read_text(encoding="utf-8"))
        time.sleep(1)
print("Server is up. Log:", ROOT / "server.log")

if IN_COLAB:
    from google.colab import output
    output.serve_kernel_port_as_window(PORT, anchor_text="Open LayaMed in a new tab")
    output.serve_kernel_port_as_iframe(PORT, height=900)
else:
    print(f"Open http://127.0.0.1:{PORT}")

In [ ]:
# 8b. Test the API without the interface
body = json.dumps({"text": "History of present illness:\n" + EXAMPLES[0][0] + "\n\nPhysical examination:\n" + EXAMPLES[0][1]
                   + "\n\nLabs:\n" + EXAMPLES[0][2]}).encode()
req = urllib.request.Request(f"http://127.0.0.1:{PORT}/api/analyze", body, {"Content-Type": "application/json"})
r = json.loads(urllib.request.urlopen(req).read())
print(r["state"], "|", r.get("leader"), {k: round(v, 3) for k, v in (r.get("probabilities") or {}).items()})

## 9. Take the model to your computer (optional)

The cell below zips `checkpoint/`, `data/scope.pt` and `data/recalibration.json` (~1.3 GB). Unzip it in the cloned repository folder and follow `docs/08-run-locally.md`. To stop the Colab server: `server.terminate()`.

In [ ]:
# 9. Package for download
import zipfile

package = ROOT / "layamed_model.zip"
with zipfile.ZipFile(package, "w", zipfile.ZIP_STORED) as z:
    for p in sorted(CHECKPOINT.rglob("*")):
        if p.is_file():
            z.write(p, p.relative_to(ROOT))
    for p in [ROOT / "data" / "scope.pt", ROOT / "data" / "recalibration.json"]:
        if p.exists():
            z.write(p, p.relative_to(ROOT))
print(f"{package} ({package.stat().st_size / 1e9:.2f} GB)")
if IN_COLAB:
    from google.colab import files
    files.download(str(package))